# Notebook 3 — Destilação (E3): rotulagem de sequência sobre o BIO

Projeto **Mineração de Argumentação em Community Notes BR** (PLN — UFSCar 2026/1 · Profa. Dra. Helena Caseli · Álvaro Barros de Carvalho, Davi Machado da Rocha · [repositório](https://github.com/rocdav/community-notes-br-argmining/)). Consome o dataset BIO do `notebook_conclusao.ipynb`. Desenvolvido originalmente como Projeto 2 de IA (ICMC-USP, Prof. Dr. Ricardo Marcondes Marcacini), incorporado aqui como a estratégia **E3**.

> **Resumo.** Com apenas 60 notas de gold humano (**consenso adjudicado**), usamos as anotações do LLM **E2** como supervisão *silver* e reservamos o gold para o teste final. Comparamos Naive Bayes, HMM, regressão logística, CRF, BERTimbau e dois decodificadores estruturados sobre as mesmas emissões congeladas (Viterbi IOB2 fixo e CRF de transições aprendidas), mais uma ablação de **professor** (silver do **E2b**, modelo aberto local). Melhor aluno: «modelo», F1 estrita «0,xxx»; professor E2: «0,xxx». *(Preencher da §8 após a execução.)*

## Em uma página

**Problema.** Marcar automaticamente, token a token (esquema **BIO**: `B-`/`I-`/`O` sobre CLAIM, EVIDENCIA, FONTE, QUALIFICADOR → 9 classes), os componentes argumentativos de cada nota. O desafio concentra-se nas **fronteiras** dos trechos.

**Ideia.** Já temos dois anotadores automáticos: **E1** (regras, ~ms/nota) e **E2** (LLM, ~s/nota). **Destilamos** o E2: os alunos treinam nas marcações dele (*silver*: notas não-meta fora do gold) e são medidos contra o **gold humano adjudicado** (60 notas, bloqueadas para escolha de hiperparâmetros — a seleção, auditável no Apêndice, usou só uma divisão de desenvolvimento *silver*).

**Percurso.** Dois eixos clássicos (generativo×discriminativo, local×sequencial): Naive Bayes → HMM → regressão logística → CRF → **BERTimbau**; sobre as mesmas emissões do BERT, três decodificadores (`argmax`, Viterbi com legalidade IOB2 fixa, CRF de transições aprendidas) isolam o efeito da estrutura de sequência. Uma ablação final troca o **professor**: silver do **E2b** (mesmo protocolo do E2, modelo aberto local).

**Reprodução.** Executar de cima para baixo; BERT requer GPU (Colab). `SEED = 42`; operações de GPU podem produzir pequenas variações. Métricas na §4: F1 estrita IOB2 (`seqeval`), F1 relaxada com pareamento 1:1 e macro-F1 de token.

## 1. Definição do problema

Aprendizado supervisionado de **rotulagem de sequência** (*token classification*): um rótulo BIO por token, 9 classes. Exemplo:

```
O/O  gesto/B-CLAIM  de/I-CLAIM  Musk/I-CLAIM  não/I-CLAIM  é/I-CLAIM  polêmico/I-CLAIM
,/O  pois/O  ele/B-EVIDENCIA  faz/I-EVIDENCIA  a/I-EVIDENCIA  simulação/I-EVIDENCIA
```

Interessa para ML por três razões: **dependência entre rótulos vizinhos** (`I-X` só faz sentido após `B-X`/`I-X`), **classes desbalanceadas** (FONTE textual e QUALIFICADOR raros) e fronteiras de span subjetivas. A pergunta prática: *dá para destilar o E2 (~segundos/nota) num aluno barato (~ms/nota) que recupere a estrutura argumentativa humana melhor que a heurística E1?*

## 2. Base de dados

`data/dataset_anotado_final_com_bio.csv` — arquivo canônico do repositório (1.901 notas × 36 colunas). Colunas usadas: `bio_tokens_json` (tokenização canônica), `e1/e2/e2b_span_bio_json` (BIO das estratégias), `humano_span_bio_json` (**gold adjudicado**, 60 notas), `is_meta`, `e1_total_ms`/`e2_ms`/`e2b_ms` (custo). Supervisão de treino = E2 (*silver*); teste = gold humano. A seção 2.1 recalcula o retrato do arquivo direto do CSV — nenhum número deste texto precisa ser confiado às cegas.

**Limitações herdadas:** gold pequeno (60 notas), treino com o ruído do professor, tipos raros com pouco suporte.

In [ ]:
# ====== 0. Setup ======
import subprocess, sys, importlib.metadata as _md

def _ver(pkg):
    try: return _md.version(pkg)
    except Exception: return None

# Extras leves do projeto (nao tocam nos pacotes pesados ja presentes no Colab).
# pytorch-crf (import: torchcrf) e a cabeca CRF do Modelo D: e puro-Python e nao
# altera torch/transformers, entao nao ha conflito de versoes nem ImportError.
_extras = ["sklearn-crfsuite", "seqeval", "pytorch-crf"]
_faltam = [p for p in _extras if _ver(p) is None]
if _faltam:
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed"] + _faltam
    print("Instalando:", " ".join(_faltam))
    subprocess.run(cmd, check=True)

# Diagnostico do ambiente
print("\nVersoes no ambiente:")
for p in ["torch", "transformers", "tokenizers", "accelerate", "datasets",
          "seqeval", "sklearn-crfsuite", "pytorch-crf"]:
    print(f"  {p:<18} {_ver(p)}")

# Verificacao de sanidade: confirma que os imports criticos funcionam SEM crash.
_erros = []
try:
    from transformers import Trainer  # noqa  (dispara o import lazy usado no treino)
except Exception as e:
    _erros.append(f"transformers.Trainer: {e}")
try:
    from torchcrf import CRF  # noqa  (cabeca CRF do Modelo D)
except Exception as e:
    _erros.append(f"torchcrf (pytorch-crf): {e}")
if _erros:
    print("\n*** Import critico falhou ***")
    for m in _erros: print("  -", m)
    raise SystemExit("Resolva o import acima antes de seguir.")
print("\nAmbiente OK - imports criticos validados, pode seguir o notebook.")

# ---------- imports base e seed ----------
import os, json, time, random, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
except Exception:
    pass

# ---------- localizacao do dataset (repo-first) ----------
CSV_NAME = "dataset_anotado_final_com_bio.csv"
CSV_CANDIDATES = [
    os.path.join("..", "data", CSV_NAME),   # rodando dentro do repo (notebooks/)
    os.path.join("data", CSV_NAME),
    os.path.join("dados", CSV_NAME),
    CSV_NAME,
]
CSV_PATH = next((p for p in CSV_CANDIDATES if os.path.exists(p)), None)
if CSV_PATH is None:
    os.makedirs("data", exist_ok=True)
    CSV_PATH = os.path.join("data", CSV_NAME)
    print("Baixando dataset do repositorio (GitHub raw)...")
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/rocdav/community-notes-br-argmining/main/data/" + CSV_NAME,
        CSV_PATH)

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Nao foi possivel localizar ou baixar {CSV_NAME}")

print("CSV:", os.path.abspath(CSV_PATH))


In [ ]:
# ====== 1. Carregamento dos dados ======
def loads(s):
    if not isinstance(s, str): return []
    s = s.strip()
    if s in ("", "[]", "null", "None", "nan"): return []
    try: return json.loads(s)
    except Exception: return []

def as_bool(value):
    if isinstance(value, str):
        return value.strip().lower() in {"true", "1", "yes", "sim"}
    return bool(value) if pd.notna(value) else False

df = pd.read_csv(CSV_PATH, encoding="utf-8")

def build_rows(frame):
    rows = []
    for _, r in frame.iterrows():
        toks = loads(r["bio_tokens_json"])
        if not toks:
            continue
        rows.append(dict(
            noteId=r["noteId"],
            text=r.get("text", ""),
            is_meta=as_bool(r["is_meta"]),
            tokens=toks,
            e1=loads(r["e1_span_bio_json"]),
            e2=loads(r["e2_span_bio_json"]),
            e2b=loads(r.get("e2b_span_bio_json")),
            gold=loads(r["humano_span_bio_json"]),
            e2_ms=float(r["e2_ms"]) if pd.notna(r["e2_ms"]) else np.nan,
            e1_ms=float(r["e1_ms"]) if pd.notna(r["e1_ms"]) else np.nan,
        ))
    return rows

rows = build_rows(df)
print(f"notas com tokens: {len(rows)} | colunas: {len(df.columns)}")


## 2.1 Visão inicial do conjunto de dados

Antes de dividir os dados, verificamos quatro pontos: **volume**, **alinhamento das sequências**, **cobertura das anotações** e **desbalanceamento das classes**. Essa inspeção é descritiva e não usa o teste para escolher modelos ou parâmetros.

**O que esperar.** A classe `O` é majoritária; CLAIM e EVIDENCIA concentram a maior parte dos spans; FONTE e, sobretudo, QUALIFICADOR têm pouco suporte no *gold*. Essa assimetria será retomada na interpretação das métricas por classe.


In [ ]:
# ====== 1.1 Visao inicial do dataset ======
from collections import Counter
import matplotlib.pyplot as plt

def alinhada(row, key):
    return bool(row[key]) and len(row[key]) == len(row["tokens"])

resumo_dados = pd.DataFrame([
    ("notas no CSV", len(df)),
    ("notas com tokens validos", len(rows)),
    ("tokens", sum(len(r["tokens"]) for r in rows)),
    ("meta-notas", sum(r["is_meta"] for r in rows)),
    ("sequencias E2 alinhadas", sum(alinhada(r, "e2") for r in rows)),
    ("E2 com ao menos um span", sum(alinhada(r, "e2") and any(t != "O" for t in r["e2"]) for r in rows)),
    ("sequencias E2b alinhadas", sum(alinhada(r, "e2b") for r in rows)),
    ("sequencias gold alinhadas", sum(alinhada(r, "gold") for r in rows)),
    ("gold com ao menos um span", sum(alinhada(r, "gold") and any(t != "O" for t in r["gold"]) for r in rows)),
], columns=["indicador", "valor"])
display(resumo_dados)

def contagem_componentes(sequencias):
    contador = Counter()
    for seq in sequencias:
        for tag in seq:
            contador["O" if tag == "O" else tag.split("-", 1)[1]] += 1
    return contador

e2_cont = contagem_componentes([r["e2"] for r in rows if alinhada(r, "e2")])
gold_cont = contagem_componentes([r["gold"] for r in rows if alinhada(r, "gold")])
tipos = ["O", "CLAIM", "EVIDENCIA", "FONTE", "QUALIFICADOR"]
dist_classes = pd.DataFrame({
    "componente": tipos,
    "tokens E2": [e2_cont[t] for t in tipos],
    "tokens gold": [gold_cont[t] for t in tipos],
})
dist_classes["E2 (%)"] = (100 * dist_classes["tokens E2"] / dist_classes["tokens E2"].sum()).round(2)
dist_classes["gold (%)"] = (100 * dist_classes["tokens gold"] / dist_classes["tokens gold"].sum()).round(2)
display(dist_classes)

amostra_cols = [c for c in ["noteId", "text", "consenso", "macrotheme_label"] if c in df.columns]
amostra = df.loc[~df["is_meta"].map(as_bool), amostra_cols].dropna(subset=["text"]).head(3).copy()
amostra["text"] = amostra["text"].str.replace(r"\s+", " ", regex=True).str.slice(0, 180)
display(amostra)

ax = dist_classes.set_index("componente")[["E2 (%)", "gold (%)"]].plot(
    kind="bar", figsize=(8, 3.6), rot=0
)
ax.set_ylabel("% dos tokens")
ax.set_title("Distribuicao dos componentes: supervisao silver vs gold humano")
plt.tight_layout()
plt.show()


## 3. Pré-processamento e estratégia de divisão

| Etapa | Decisão | Justificativa |
|---|---|---|
| Tokenização | reutilizar a tokenização canônica do CSV | mantém E1, E2 e humano nos mesmos offsets |
| Sanidade | aceitar apenas sequências BIO alinhadas aos tokens | evita avaliação desalinhada |
| Exclusão | remover meta-notas do treino | não são textos argumentativos reais |
| Treino | notas não-meta, fora do gold, rotuladas por E2 | supervisão silver em escala |
| Teste | 60 notas com anotação humana | referência final bloqueada |
| Neural | máximo de 256 sub-tokens | controla memória; palavras não vistas pelo encoder recebem `O` na reconstrução |

As configurações foram fixadas a partir de uma divisão determinística de desenvolvimento com 12% do *silver*. O apêndice permite auditar essa seleção sem consultar o gold. Para o BERT, o ranking de taxas próximas pode variar entre execuções de GPU.

**Código a seguir:** cria treino e teste, confirma seus tamanhos e lista as classes BIO.


In [ ]:
# ====== 2. Split treino (silver E2) / teste (gold humano) ======
def split(rows):
    gold = [r for r in rows if r["gold"] and len(r["gold"])==len(r["tokens"])]
    gold_ids = {r["noteId"] for r in gold}
    train = [r for r in rows
             if (not r["is_meta"]) and r["noteId"] not in gold_ids
             and r["e2"] and len(r["e2"])==len(r["tokens"])]
    return train, gold

train, test = split(rows)
LABELS = sorted({t for r in rows for t in r["e2"]} | {"O"})
print(f"treino (silver E2): {len(train)} notas, {sum(len(r['tokens']) for r in train)} tokens")
print(f"teste  (gold human): {len(test)} notas, {sum(len(r['tokens']) for r in test)} tokens")
print("classes:", LABELS)


## 3.1 Protocolo experimental

O protocolo separa:

| Papel | Rótulo | Número de notas | Uso |
|---|---|---:|---|
| treino completo | E2 (*silver*) | 1.437 | treinamento final |
| `train_fit` | E2 (*silver*) | 1.265 | treinamento durante a auditoria da busca |
| desenvolvimento | E2 (*silver*) | 172 | comparação de configurações |
| teste | humano (*gold* **adjudicado**) | 60 | avaliação final |

O gold não participa da escolha de hiperparâmetros. Os modelos principais usam as configurações registradas em `BEST` e são treinados sobre todo o silver.

No bloco neural, um único BERTimbau gera as emissões usadas por `argmax`, Viterbi fixo e CRF aprendido. Isso evita atribuir ao decodificador diferenças provocadas por um segundo fine-tuning do encoder.


## 4. Métricas de avaliação

Reportamos três níveis, todos calculados contra o **gold humano**:

1. **F1 de token (macro)** — média simples da F1 das nove classes BIO, incluindo `O`.
2. **F1 de span relaxada** — um span predito e um span gold do mesmo tipo podem formar um acerto quando se sobrepõem. O pareamento é **um-para-um**: cada span participa de, no máximo, um verdadeiro positivo.
3. **F1 de span estrita IOB2** — calculada pelo `seqeval` com `mode="strict"` e `scheme=IOB2`; tipo, início e fim precisam coincidir exatamente, e sequências BIO malformadas não são normalizadas silenciosamente.

> A macro-F1 de token inclui a classe `O`, portanto deve ser lida junto das métricas de span. A F1 relaxada mede recuperação aproximada de conteúdo, não qualidade exata de fronteira.

Também registramos custo de inferência. Para os modelos neurais, o tempo do BERT corresponde ao encoder mais `argmax`; os tempos de Viterbi e CRF medem apenas o **acréscimo do decodificador** sobre emissões já calculadas e não são latências fim a fim diretamente comparáveis. A seção de *bootstrap* reamostra as 60 notas de teste e quantifica incerteza **condicionada às predições desta execução**; ela não inclui variação causada por novas sementes ou novos treinamentos.

**Código a seguir:** define a avaliação estrita IOB2, a avaliação relaxada com pareamento um-para-um e a macro-F1 de token.

In [ ]:
# ====== 3. Funções de avaliação ======
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_p, recall_score as seq_r
from seqeval.metrics import classification_report as seq_report
from seqeval.scheme import IOB2
from sklearn.metrics import f1_score as sk_f1

SEQEVAL_STRICT = {"mode": "strict", "scheme": IOB2}

def strict_f1(y_true, y_pred):
    """F1 de span estrita, sem normalizar sequencias BIO invalidas."""
    return seq_f1(y_true, y_pred, **SEQEVAL_STRICT)

def spans_from_bio(bio):
    """Reconstrói spans (tipo, início, fim) segundo IOB2 estrito.

    Apenas B-X abre um span e apenas I-X imediatamente após B-X/I-X do mesmo
    tipo o continua. Um I-X ilegal não é convertido implicitamente em B-X.
    """
    out, i, n = [], 0, len(bio)
    while i < n:
        tag = bio[i]
        if tag.startswith("B-"):
            tipo = tag[2:]
            j = i + 1
            while j < n and bio[j] == "I-" + tipo:
                j += 1
            out.append((tipo, i, j))
            i = j
        else:
            i += 1
    return out

def _max_matching_overlap(gold_spans, pred_spans):
    """Cardinalidade do pareamento bipartido máximo entre spans sobrepostos."""
    adj = []
    for tipo_p, ini_p, fim_p in pred_spans:
        adj.append([
            gi for gi, (tipo_g, ini_g, fim_g) in enumerate(gold_spans)
            if tipo_p == tipo_g and ini_p < fim_g and ini_g < fim_p
        ])

    matched_gold = [-1] * len(gold_spans)

    def augment(pred_i, seen):
        for gold_i in adj[pred_i]:
            if gold_i in seen:
                continue
            seen.add(gold_i)
            if matched_gold[gold_i] == -1 or augment(matched_gold[gold_i], seen):
                matched_gold[gold_i] = pred_i
                return True
        return False

    return sum(augment(pred_i, set()) for pred_i in range(len(pred_spans)))

def relaxed_prf(y_true, y_pred):
    """F1 relaxada com sobreposicao e pareamento um-para-um por nota."""
    tp = total_pred = total_gold = 0
    for gold_bio, pred_bio in zip(y_true, y_pred):
        gold_spans = spans_from_bio(gold_bio)
        pred_spans = spans_from_bio(pred_bio)
        tp += _max_matching_overlap(gold_spans, pred_spans)
        total_pred += len(pred_spans)
        total_gold += len(gold_spans)
    precision = tp / total_pred if total_pred else 0.0
    recall = tp / total_gold if total_gold else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

def token_macro_f1(y_true, y_pred):
    """F1 macro no nível de token; inclui a classe O."""
    flat_true = [tag for seq in y_true for tag in seq]
    flat_pred = [tag for seq in y_pred for tag in seq]
    return sk_f1(flat_true, flat_pred, average="macro", zero_division=0)

def avaliar(nome, y_true, y_pred, mostrar=True):
    res = dict(
        modelo=nome,
        f1_estrita=strict_f1(y_true, y_pred),
        p_estrita=seq_p(y_true, y_pred, **SEQEVAL_STRICT),
        r_estrita=seq_r(y_true, y_pred, **SEQEVAL_STRICT),
    )
    rp, rr, rf = relaxed_prf(y_true, y_pred)
    res.update(f1_relaxada=rf, token_macro_f1=token_macro_f1(y_true, y_pred))
    if mostrar:
        print(f"\n### {nome}")
        print("  span estrita  P/R/F1: %.3f / %.3f / %.3f" %
              (res["p_estrita"], res["r_estrita"], res["f1_estrita"]))
        print("  span relaxada P/R/F1: %.3f / %.3f / %.3f" % (rp, rr, rf))
        print("  token macro-F1: %.3f" % res["token_macro_f1"])
        print(seq_report(y_true, y_pred, digits=3, **SEQEVAL_STRICT))
    return res

resultados = []
y_test = [r["gold"] for r in test]


## 5. Referências: heurística **E1** e professor **E2**

Antes de treinar qualquer modelo, medimos as duas fontes automáticas contra o gold:
- **E1 (regras)** é uma referência simples a ser superada e erra fronteiras sistematicamente.
- **E2 (LLM)** é o **professor** que produz a supervisão *silver* e uma referência forte de qualidade contra o humano.

E2 não é um teto teórico: um aluno pode, em princípio, regularizar ou corrigir parte do ruído do professor. Neste experimento, porém, E2 permanece com a maior F1 estrita observada.

**Código a seguir:** avalia E1 e E2 diretamente contra o gold (sem nenhum treino).

In [ ]:
# ====== 4. Referências E1 e E2 (sem treino) ======
resultados.append(avaliar("E1 (regras)",        y_test, [r["e1"] for r in test]))
resultados.append(avaliar("E2 (LLM, teacher)",  y_test, [r["e2"] for r in test]))
if any(alinhada(r, "e2b") for r in test):
    resultados.append(avaliar("E2b (LLM local)", y_test,
        [r["e2b"] if alinhada(r, "e2b") else ["O"] * len(r["tokens"]) for r in test]))


## 6. Os paradigmas de rotulagem de sequência

A rotulagem de sequência pode ser organizada por dois eixos clássicos:

- **Generativo × discriminativo** — modelar a distribuição conjunta dos dados e rótulos ou modelar diretamente o rótulo condicionado ao texto.
- **Local × sequencial** — decidir cada token separadamente ou atribuir a cadeia completa modelando dependências entre saídas.

| | **local** | **sequencial** |
|---|---|---|
| **generativo** | Naive Bayes | HMM |
| **discriminativo** | Regressão logística | CRF |

Em seguida avaliamos o BERTimbau com classificação local e dois decodificadores aplicados às **mesmas emissões do BERT**:

- `argmax`, sem transições;
- Viterbi com legalidade BIO fixa;
- CRF que aprende apenas parâmetros de início, fim e transição, mantendo BERT e emissões congelados.

Esse desenho torna controlada a comparação entre os três decodificadores. Já as comparações entre famílias clássicas e o BERT continuam sendo comparações empíricas entre modelos distintos.

> As configurações em `BEST` foram fixadas a partir do desenvolvimento *silver*. A busca opcional documenta esse processo, mas resultados neurais ainda podem variar entre execuções de GPU.


In [ ]:
# ====== Configuracoes fixadas a partir do desenvolvimento silver ======
REPRODUZIR_BUSCA = False
BEST = {
    "nb":      dict(addk=0.1,  min_count=1),
    "hmm":     dict(addk=0.01, min_count=1),
    "logreg":  dict(C=0.1, class_weight=None),
    "crf":     dict(c1=1.0, c2=0.05),
    "bert_lr": 5e-5,
}
BERT_EPOCHS = 4
CRF_DECODER = {
    "epochs": 40,
    "lr": 5e-2,
    "weight_decay": 1e-4,
    "batch_size": 64,
}
print("Config usada:", BEST)
print("Epocas BERT:", BERT_EPOCHS, "| decoder CRF:", CRF_DECODER)
print("REPRODUZIR_BUSCA =", REPRODUZIR_BUSCA)


### 6.1 Generativo local — Naive Bayes

> **Intuição.** O canto mais simples do quadro: para cada palavra, escolhe o rótulo que maximiza **P(rótulo)·P(palavra|rótulo)**, estimados por **contagem** no treino *silver*. É *generativo* (modela como a palavra é gerada por cada rótulo) e *local* (decide cada token sozinho, sem olhar os vizinhos). Sem contexto nem estrutura de sequência, tende a produzir spans fragmentados e BIO malformado.

In [ ]:
# ====== Naive Bayes (gerativo local) ======
# P(rótulo)·P(palavra|rótulo) por contagem (suavização add-k), decisão por TOKEN.
import numpy as np
from collections import Counter

def _contagens(subset, addk, min_count):
    """Estima emissão/prior/inicial/transição por contagem suavizada (add-k) sobre `subset`.
    Palavras com frequência < min_count viram <UNK> (trata OOV no teste)."""
    freq = Counter(w.lower() for r in subset for w in r["tokens"])
    vocab = {w for w, c in freq.items() if c >= min_count}
    V = sorted(vocab | {"<UNK>"}); v2i = {w: i for i, w in enumerate(V)}
    def wid(w):
        w = w.lower(); return w if w in vocab else "<UNK>"
    K = len(LABELS); l2i = {l: i for i, l in enumerate(LABELS)}
    emis = np.full((K, len(V)), addk); prior = np.full(K, addk)
    init = np.full(K, addk); trans = np.full((K, K), addk)
    for r in subset:
        labs, toks = r["e2"], r["tokens"]; init[l2i[labs[0]]] += 1
        for t, (w, l) in enumerate(zip(toks, labs)):
            i = l2i[l]; emis[i, v2i[wid(w)]] += 1; prior[i] += 1
            if t > 0: trans[l2i[labs[t-1]], i] += 1
    return wid, v2i, (np.log(emis / emis.sum(1, keepdims=True)),
                      np.log(prior / prior.sum()),
                      np.log(init / init.sum()),
                      np.log(trans / trans.sum(1, keepdims=True)))

wid, v2i, (log_emis, log_prior, log_init, log_trans) = _contagens(train, **BEST["nb"])
t0 = time.time()
nb_pred = [[LABELS[int((log_prior + log_emis[:, v2i[wid(w)]]).argmax())] for w in r["tokens"]]
           for r in test]
nb_ms = 1000 * (time.time() - t0) / len(test)
resultados.append({**avaliar("Naive Bayes", y_test, nb_pred), "ms_por_nota": nb_ms})


### 6.2 Generativo sequencial — HMM

> **Intuição.** O **modelo oculto de Markov (HMM)** acrescenta ao Naive Bayes a peça que faltava: **transições** P(rótulo | rótulo anterior). Continua generativo, mas agora modela a **sequência** e decodifica a cadeia inteira por **Viterbi**. As transições, contadas dos dados, já desencorajam quase toda transição ilegal (ex.: `O → I-CLAIM` quase nunca ocorre). É o salto **local → sequencial** dentro do paradigma generativo.


In [ ]:
# ====== HMM (gerativo sequencial) ======
# Naive Bayes + transições P(rótulo|rótulo anterior); decodifica a sequência por Viterbi.
wid, v2i, (log_emis, log_prior, log_init, log_trans) = _contagens(train, **BEST["hmm"])

def hmm_viterbi(toks):
    obs = [v2i[wid(w)] for w in toks]; T = len(obs); K = len(LABELS)
    dp = np.full((T, K), -1e18); bp = np.zeros((T, K), dtype=int)
    dp[0] = log_init + log_emis[:, obs[0]]
    for t in range(1, T):
        sc = dp[t-1][:, None] + log_trans           # transição do passo anterior
        bp[t] = sc.argmax(0); dp[t] = sc.max(0) + log_emis[:, obs[t]]
    seq = [int(dp[-1].argmax())]
    for t in range(T-1, 0, -1): seq.append(int(bp[t, seq[-1]]))
    return [LABELS[i] for i in reversed(seq)]

t0 = time.time(); hmm_pred = [hmm_viterbi(r["tokens"]) for r in test]
hmm_ms = 1000 * (time.time() - t0) / len(test)
resultados.append({**avaliar("HMM", y_test, hmm_pred), "ms_por_nota": hmm_ms})

# transições de maior probabilidade aprendidas por contagem (devem favorecer BIO legal)
K = len(LABELS)
_pares = [(LABELS[i], LABELS[j], float(log_trans[i, j])) for i in range(K) for j in range(K)]
print("\nHMM — transições de maior probabilidade (log):")
for a, b, w in sorted(_pares, key=lambda x: -x[2])[:6]:
    print(f"  {a:>14} -> {b:<14} {w:+.2f}")


### 6.3 Discriminativo local — Regressão logística

> **Intuição.** O par *discriminativo* do Naive Bayes: também decide **uma palavra de cada vez**, mas em vez de modelar como a palavra é gerada, aprende **diretamente** o rótulo a partir de *features* (sufixos, formato, maiúsculas, pista de URL, palavras vizinhas). Continua *local* — sem encadear rótulos —, então tende a **fragmentar os spans**, mas costuma superar o Naive Bayes no mesmo nível.

**Código a seguir:** monta as *features*, treina a regressão logística (config de `BEST`) nos rótulos silver do E2 e avalia no gold.

In [ ]:
# ====== Regressão Logística (discriminativo local) — features + treino ======
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

def shape(w):
    s=[]
    for ch in w:
        s.append("X" if ch.isupper() else "x" if ch.islower() else "d" if ch.isdigit() else ch)
    return "".join(s[:8])

def tok_feats(tokens, i):
    w = tokens[i]
    f = {"bias":1.0, "w.lower":w.lower(), "w.suf3":w[-3:].lower(), "w.suf2":w[-2:].lower(),
         "w.pre3":w[:3].lower(), "w.isupper":w.isupper(), "w.istitle":w.istitle(),
         "w.isdigit":w.isdigit(), "w.len":len(w), "w.shape":shape(w),
         "w.haspunct":any(not c.isalnum() for c in w),
         "w.url":int(any(t in w.lower() for t in ("http","www",".com",".br","/")))}
    if i>0:
        p=tokens[i-1]; f.update({"-1.lower":p.lower(),"-1.istitle":p.istitle(),"-1.suf3":p[-3:].lower()})
    else: f["BOS"]=True
    if i<len(tokens)-1:
        n=tokens[i+1]; f.update({"+1.lower":n.lower(),"+1.istitle":n.istitle(),"+1.suf3":n[-3:].lower()})
    else: f["EOS"]=True
    return f

def to_feats(rows):
    return [[tok_feats(r["tokens"],i) for i in range(len(r["tokens"]))] for r in rows]

Xtr = to_feats(train); ytr = [r["e2"] for r in train]
Xte = to_feats(test)

dv = DictVectorizer(sparse=True)
Xtr_flat = dv.fit_transform([t for s in Xtr for t in s])
ytr_flat = [t for s in ytr for t in s]

lr = LogisticRegression(max_iter=1000, **BEST["logreg"])
t0=time.time(); lr.fit(Xtr_flat, ytr_flat); base_train_s=time.time()-t0
t0=time.time(); base_pred = [list(lr.predict(dv.transform(s))) for s in Xte]
base_ms = 1000*(time.time()-t0)/len(test)
print(f"tempo de treino: {base_train_s:.2f} s | inferencia: {base_ms:.3f} ms/nota")
resultados.append({**avaliar("Regressão Logística", y_test, base_pred), "ms_por_nota": base_ms})


### 6.4 Discriminativo sequencial — CRF (Conditional Random Field)

> **Intuição.** O CRF une os dois eixos: é discriminativo como a regressão logística e sequencial como o HMM. Em vez de decidir token a token, escolhe a **sequência inteira de rótulos**, aprendendo conjuntamente pesos de atributos e **transições** plausíveis (`B-CLAIM → I-CLAIM`, por exemplo). É a alternativa clássica mais diretamente voltada a fronteiras e coerência BIO.

**Código a seguir:** treina o CRF (configuração de `BEST`) nas mesmas *features* da regressão logística e imprime as transições mais fortes que ele aprendeu.

In [ ]:
# ====== 5B. CRF ======
import sklearn_crfsuite
crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=BEST["crf"]["c1"], c2=BEST["crf"]["c2"],
                           max_iterations=120, all_possible_transitions=True)
t0=time.time(); crf.fit(Xtr, ytr); crf_train_s=time.time()-t0
t0=time.time(); crf_pred = crf.predict(Xte)
crf_ms = 1000*(time.time()-t0)/len(test)
print(f"tempo de treino: {crf_train_s:.2f} s | inferencia: {crf_ms:.3f} ms/nota")
resultados.append({**avaliar("CRF", y_test, crf_pred), "ms_por_nota": crf_ms})

# transições mais prováveis aprendidas (interpretabilidade)
from collections import Counter
trans = Counter(crf.transition_features_).most_common(8)
print("\nTransições mais fortes aprendidas:")
for (a,b),w in trans: print(f"  {a:>14} -> {b:<14} {w:+.2f}")


## 7. Paradigma neural profundo — BERTimbau

> **Intuição.** Saímos das *features* manuais para **representações aprendidas**: uma rede **Transformer** pré-treinada em português (`neuralmind/bert-base-portuguese-cased`), que lê a frase inteira com contexto bidirecional. A decisão volta a ser **local** (um `argmax` por token sobre saídas riquíssimas); as próximas seções recolocam a estrutura de sequência sobre ela. **Requer GPU (Colab).**

Fazemos *fine-tuning* para classificar cada token; rótulo no 1º sub-token de cada palavra (`-100` nos demais). Usa a taxa de aprendizado selecionada (`BEST["bert_lr"]`) e `BERT_EPOCHS` épocas; treina no silver do E2 e avalia no gold.

**Código a seguir:** prepara os dados e treina o BERT (~4 min em GPU); a célula seguinte faz a predição e a avaliação.

In [ ]:
# ====== 5C. BERTimbau (rodar no Colab com GPU) ======
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          TrainingArguments, Trainer, DataCollatorForTokenClassification)

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
label2id = {l:i for i,l in enumerate(LABELS)}; id2label = {i:l for l,i in label2id.items()}
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_ds(rows, label_key):
    return Dataset.from_dict({"tokens":[r["tokens"] for r in rows],
                              "labels_bio":[r[label_key] for r in rows]})
ds_tr = make_ds(train, "e2"); ds_te = make_ds(test, "gold")

def tok_align(batch):
    enc = tok(batch["tokens"], is_split_into_words=True, truncation=True, max_length=256)
    all_labels=[]
    for i, labs in enumerate(batch["labels_bio"]):
        word_ids = enc.word_ids(batch_index=i); prev=None; ids=[]
        for wid in word_ids:
            if wid is None: ids.append(-100)
            elif wid!=prev: ids.append(label2id[labs[wid]])
            else: ids.append(-100)
            prev=wid
        all_labels.append(ids)
    enc["labels"]=all_labels; return enc

ds_tr = ds_tr.map(tok_align, batched=True, remove_columns=ds_tr.column_names)
ds_te = ds_te.map(tok_align, batched=True, remove_columns=ds_te.column_names)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id)
collator = DataCollatorForTokenClassification(tok)

args = TrainingArguments(output_dir="bert_out", learning_rate=BEST["bert_lr"],
    per_device_train_batch_size=16, num_train_epochs=BERT_EPOCHS, weight_decay=0.01,
    logging_steps=50, report_to="none", seed=SEED)
trainer = Trainer(model=model, args=args, train_dataset=ds_tr,
                  data_collator=collator, processing_class=tok)
t0=time.time(); trainer.train(); train_min=(time.time()-t0)/60
print(f"treino BERT: {train_min:.1f} min")


In [ ]:
# ====== 5C (cont.) logits BERT -> emissoes por palavra -> avaliacao ======
import numpy as np

def logits_por_palavra(logits_rows, rows):
    """Extrai a emissao do primeiro sub-token e preserva o indice da palavra."""
    saida = []
    for row_logits, r in zip(logits_rows, rows):
        enc = tok([r["tokens"]], is_split_into_words=True, truncation=True, max_length=256)
        word_ids = enc.word_ids(0)
        emissoes, posicoes, anterior = [], [], None
        for j, word_id in enumerate(word_ids):
            if word_id is None or word_id == anterior:
                anterior = word_id
                continue
            emissoes.append(row_logits[j])
            posicoes.append(word_id)
            anterior = word_id
        saida.append((np.asarray(emissoes, dtype=np.float32), posicoes))
    return saida

# Emissoes do treino alimentam somente o decoder CRF; o encoder nao sera retreinado.
train_logits = trainer.predict(ds_tr).predictions
train_word_logits = logits_por_palavra(train_logits, train)

t0 = time.time()
test_logits = trainer.predict(ds_te).predictions
test_word_logits = logits_por_palavra(test_logits, test)

bert_pred = []
for r, (emissoes, posicoes) in zip(test, test_word_logits):
    seq = ["O"] * len(r["tokens"])
    for pos, label_id in zip(posicoes, np.argmax(emissoes, axis=1)):
        seq[pos] = id2label[int(label_id)]
    bert_pred.append(seq)

bert_ms = 1000 * (time.time() - t0) / len(test)
resultados.append({**avaliar("BERTimbau", y_test, bert_pred), "ms_por_nota": bert_ms})


## 7.1 Decodificação com transições fixas: BERT + Viterbi

Esta é uma ablação controlada: reutilizamos exatamente as emissões do BERTimbau e substituímos o `argmax` por Viterbi com uma máscara que proíbe inícios e transições ilegais em IOB2.

Não há novo treinamento. Assim, a diferença entre BERTimbau e BERT+Viterbi mede apenas o efeito de impor legalidade BIO à decodificação desta execução.


In [ ]:
# ====== 5C (extra) BERT + Viterbi com legalidade IOB2 fixa ======
import numpy as np

def matriz_transicao_bio(labels):
    """Mascara em log: 0 para transicao legal e penalidade grande para ilegal."""
    n = len(labels)
    neg = -1e9
    trans = np.zeros((n, n), dtype=np.float32)
    for i, anterior in enumerate(labels):
        for j, atual in enumerate(labels):
            if atual.startswith("I-"):
                tipo = atual[2:]
                if anterior not in ("B-" + tipo, "I-" + tipo):
                    trans[i, j] = neg
    return trans

def viterbi(log_emis, log_trans, start_ok):
    n_tokens, n_labels = log_emis.shape
    dp = np.full((n_tokens, n_labels), -1e9, dtype=np.float32)
    bp = np.zeros((n_tokens, n_labels), dtype=int)
    dp[0] = log_emis[0] + start_ok
    for t in range(1, n_tokens):
        scores = dp[t - 1][:, None] + log_trans
        bp[t] = scores.argmax(0)
        dp[t] = scores.max(0) + log_emis[t]
    seq = [int(dp[-1].argmax())]
    for t in range(n_tokens - 1, 0, -1):
        seq.append(int(bp[t, seq[-1]]))
    return seq[::-1]

start_ok = np.array(
    [-1e9 if label.startswith("I-") else 0.0 for label in LABELS],
    dtype=np.float32,
)
fixed_transitions = matriz_transicao_bio(LABELS)

t0 = time.time()
bert_constr_pred = []
for r, (emissoes, posicoes) in zip(test, test_word_logits):
    seq = ["O"] * len(r["tokens"])
    if len(emissoes):
        decoded = viterbi(emissoes, fixed_transitions, start_ok)
        for pos, label_id in zip(posicoes, decoded):
            seq[pos] = LABELS[label_id]
    bert_constr_pred.append(seq)

viterbi_ms = 1000 * (time.time() - t0) / len(test)
resultados.append({
    **avaliar("BERT + Viterbi", y_test, bert_constr_pred),
    "ms_por_nota": viterbi_ms,
})


## 7.2 Decodificação com transições aprendidas: BERT + CRF

Para comparar transições fixas e aprendidas sem introduzir um segundo fine-tuning do encoder, congelamos as emissões produzidas pelo **mesmo BERTimbau** da seção anterior.

O CRF abaixo aprende somente:

- scores de início;
- scores de fim;
- a matriz de transição entre rótulos.

| Decodificador | Emissões | Transições |
|---|---|---|
| BERTimbau | mesmas emissões congeladas | nenhuma (`argmax`) |
| BERT + Viterbi | mesmas emissões congeladas | legalidade IOB2 fixa |
| BERT + CRF | mesmas emissões congeladas | parâmetros aprendidos no treino *silver* |

Portanto, a comparação isola o efeito do **decodificador sobre estas emissões**. Ela não demonstra superioridade geral de uma família arquitetural e permanece condicionada a uma execução do BERT.


In [ ]:
# ====== 5D. BERT + CRF sobre emissoes congeladas ======
import torch
from torch.utils.data import DataLoader
from torchcrf import CRF

device = "cuda" if torch.cuda.is_available() else "cpu"

def exemplos_emissoes(rows, word_logits, label_key=None):
    exemplos = []
    for r, (emissoes, posicoes) in zip(rows, word_logits):
        item = {
            "emissions": torch.tensor(emissoes, dtype=torch.float32),
            "positions": posicoes,
        }
        if label_key is not None:
            item["tags"] = torch.tensor(
                [label2id[r[label_key][pos]] for pos in posicoes],
                dtype=torch.long,
            )
        exemplos.append(item)
    return exemplos

def collate_emissoes(batch):
    batch_size = len(batch)
    max_words = max(len(item["positions"]) for item in batch)
    n_labels = len(LABELS)
    emissions = torch.zeros(batch_size, max_words, n_labels, dtype=torch.float32)
    mask = torch.zeros(batch_size, max_words, dtype=torch.bool)
    has_tags = "tags" in batch[0]
    tags = torch.zeros(batch_size, max_words, dtype=torch.long) if has_tags else None
    positions = []
    for i, item in enumerate(batch):
        n_words = len(item["positions"])
        emissions[i, :n_words] = item["emissions"]
        mask[i, :n_words] = True
        if has_tags:
            tags[i, :n_words] = item["tags"]
        positions.append(item["positions"])
    out = {"emissions": emissions, "mask": mask, "positions": positions}
    if has_tags:
        out["tags"] = tags
    return out

train_emissions_ds = exemplos_emissoes(train, train_word_logits, "e2")
test_emissions_ds = exemplos_emissoes(test, test_word_logits)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

learned_crf = CRF(len(LABELS), batch_first=True).to(device)
with torch.no_grad():
    learned_crf.start_transitions.zero_()
    learned_crf.end_transitions.zero_()
    learned_crf.transitions.zero_()

optimizer = torch.optim.AdamW(
    learned_crf.parameters(),
    lr=CRF_DECODER["lr"],
    weight_decay=CRF_DECODER["weight_decay"],
)
generator = torch.Generator().manual_seed(SEED)
loader = DataLoader(
    train_emissions_ds,
    batch_size=CRF_DECODER["batch_size"],
    shuffle=True,
    collate_fn=collate_emissoes,
    generator=generator,
)

t0 = time.time()
learned_crf.train()
for epoch in range(CRF_DECODER["epochs"]):
    total_loss = 0.0
    for batch in loader:
        emissions = batch["emissions"].to(device)
        tags = batch["tags"].to(device)
        mask = batch["mask"].to(device)
        loss = -learned_crf(
            emissions,
            tags,
            mask=mask,
            reduction="token_mean",
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch == 0 or (epoch + 1) % 10 == 0:
        print(
            f"decoder CRF epoca {epoch + 1:02d}/{CRF_DECODER['epochs']} "
            f"loss={total_loss / len(loader):.4f}"
        )
crf_decoder_train_s = time.time() - t0
print(f"treino apenas das transicoes CRF: {crf_decoder_train_s:.1f} s")

t0 = time.time()
learned_crf.eval()
all_decoded, all_positions = [], []
with torch.no_grad():
    test_loader = DataLoader(
        test_emissions_ds,
        batch_size=64,
        shuffle=False,
        collate_fn=collate_emissoes,
    )
    for batch in test_loader:
        decoded = learned_crf.decode(
            batch["emissions"].to(device),
            mask=batch["mask"].to(device),
        )
        all_decoded.extend(decoded)
        all_positions.extend(batch["positions"])

bert_crf_pred = []
for r, decoded, positions in zip(test, all_decoded, all_positions):
    seq = ["O"] * len(r["tokens"])
    for pos, label_id in zip(positions, decoded):
        seq[pos] = LABELS[label_id]
    bert_crf_pred.append(seq)

bert_crf_ms = 1000 * (time.time() - t0) / len(test)
resultados.append({
    **avaliar("BERT + CRF (emissões fixas)", y_test, bert_crf_pred),
    "ms_por_nota": bert_crf_ms,
})

transition_matrix = learned_crf.transitions.detach().cpu().numpy()
transicoes = [
    (LABELS[i], LABELS[j], float(transition_matrix[i, j]))
    for i in range(len(LABELS))
    for j in range(len(LABELS))
]
print("\nTransicoes de maior peso aprendidas pelo decoder CRF:")
for anterior, atual, peso in sorted(transicoes, key=lambda x: -x[2])[:8]:
    print(f"  {anterior:>14} -> {atual:<14} {peso:+.2f}")


## 7.3 Ablação de professor: e se o professor for o modelo aberto local?

Mesmo pipeline do melhor aluno (BERT com decodificador CRF sobre emissões congeladas),
trocando **apenas a supervisão**: silver do **E2b** (modelo aberto local) em vez do E2
(API proprietária). Mede quanto o aluno depende da qualidade/estilo do professor — e se um
professor 100 % aberto, local e reprodutível basta para a destilação.

In [ ]:
# ====== 5E. Ablacao de professor: BERT + CRF treinado no silver do E2b ======
RUN_E2B_TEACHER = True

if RUN_E2B_TEACHER and any(alinhada(r, "e2b") for r in train):
    train_b = [r for r in train if alinhada(r, "e2b")]
    print(f"treino silver E2b: {len(train_b)} notas")
    ds_tr_b = make_ds(train_b, "e2b").map(tok_align, batched=True,
                                          remove_columns=["tokens", "labels_bio"])
    model_b = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id)
    args_b = TrainingArguments(output_dir="bert_out_e2b", learning_rate=BEST["bert_lr"],
        per_device_train_batch_size=16, num_train_epochs=BERT_EPOCHS, weight_decay=0.01,
        logging_steps=50, report_to="none", seed=SEED)
    trainer_b = Trainer(model=model_b, args=args_b, train_dataset=ds_tr_b,
                        data_collator=collator, processing_class=tok)
    t0 = time.time(); trainer_b.train()
    print(f"treino BERT (silver E2b): {(time.time() - t0) / 60:.1f} min")

    tr_word_b = logits_por_palavra(trainer_b.predict(ds_tr_b).predictions, train_b)
    te_word_b = logits_por_palavra(trainer_b.predict(ds_te).predictions, test)
    tr_emis_b = exemplos_emissoes(train_b, tr_word_b, "e2b")
    te_emis_b = exemplos_emissoes(test, te_word_b)

    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    crf_b = CRF(len(LABELS), batch_first=True).to(device)
    with torch.no_grad():
        crf_b.start_transitions.zero_(); crf_b.end_transitions.zero_(); crf_b.transitions.zero_()
    opt_b = torch.optim.AdamW(crf_b.parameters(), lr=CRF_DECODER["lr"],
                              weight_decay=CRF_DECODER["weight_decay"])
    loader_b = DataLoader(tr_emis_b, batch_size=CRF_DECODER["batch_size"], shuffle=True,
                          collate_fn=collate_emissoes,
                          generator=torch.Generator().manual_seed(SEED))
    crf_b.train()
    for ep in range(CRF_DECODER["epochs"]):
        for batch in loader_b:
            loss = -crf_b(batch["emissions"].to(device), batch["tags"].to(device),
                          mask=batch["mask"].to(device), reduction="token_mean")
            opt_b.zero_grad(); loss.backward(); opt_b.step()

    crf_b.eval()
    dec_b, pos_b = [], []
    with torch.no_grad():
        for batch in DataLoader(te_emis_b, batch_size=64, shuffle=False,
                                collate_fn=collate_emissoes):
            dec_b.extend(crf_b.decode(batch["emissions"].to(device),
                                      mask=batch["mask"].to(device)))
            pos_b.extend(batch["positions"])
    bert_crf_e2b_pred = []
    for r, decoded, positions in zip(test, dec_b, pos_b):
        seq = ["O"] * len(r["tokens"])
        for pos, lid in zip(positions, decoded):
            seq[pos] = LABELS[lid]
        bert_crf_e2b_pred.append(seq)
    resultados.append(avaliar("BERT + CRF (teacher E2b)", y_test, bert_crf_e2b_pred))
else:
    print("Ablacao E2b desativada ou sem BIO do E2b no treino.")

## 8. Resultados consolidados

A tabela reúne referências, paradigmas clássicos e os três decodificadores neurais. Todas as métricas usam o mesmo teste humano.

Leituras controladas:

- BERTimbau × BERT+Viterbi: mesmas emissões, legalidade fixa adicionada;
- BERTimbau × BERT+CRF: mesmas emissões, transições aprendidas adicionadas;
- BERT+Viterbi × BERT+CRF: transições fixas versus aprendidas sobre as mesmas emissões.

As demais comparações envolvem modelos e procedimentos de treino diferentes e devem ser interpretadas como resultados empíricos, não como ablações causais.


In [ ]:
# ====== 6. Tabela + graficos ======
import matplotlib.pyplot as plt

ordem_modelos = [
    "E1 (regras)", "E2 (LLM, teacher)", "E2b (LLM local)",
    "Naive Bayes", "HMM",
    "Regressão Logística", "CRF",
    "BERTimbau", "BERT + Viterbi", "BERT + CRF (emissões fixas)",
    "BERT + CRF (teacher E2b)",
]
res_df = pd.DataFrame(resultados)[
    [
        "modelo", "f1_estrita", "f1_relaxada", "token_macro_f1",
        "p_estrita", "r_estrita", "ms_por_nota",
    ]
]
res_df["modelo"] = pd.Categorical(
    res_df["modelo"], categories=ordem_modelos, ordered=True
)
res_df = res_df.sort_values("modelo").reset_index(drop=True)
res_df["modelo"] = res_df["modelo"].astype(str)

print(res_df.round(3).to_string(index=False))
os.makedirs("resultados", exist_ok=True)
res_df.round(6).to_csv("resultados/comparacao_modelos.csv", index=False)

plot_df = res_df.set_index("modelo")[
    ["f1_estrita", "f1_relaxada", "token_macro_f1"]
]
ax = plot_df.plot(
    kind="bar",
    figsize=(11, 5),
    rot=24,
    color=["#315A8C", "#5E9C76", "#D18B47"],
)
ax.set_ylabel("F1")
ax.set_xlabel("")
ax.set_ylim(0, 0.65)
ax.set_title("Comparação por paradigma contra o gold humano (60 notas)")
ax.legend(["Span estrita IOB2", "Span relaxada 1:1", "Token macro"], fontsize=8)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.savefig("resultados/comparacao_modelos.png", dpi=150)
plt.show()


## 8.1 Incerteza das estimativas: *bootstrap* sobre as notas

Reamostramos as 60 notas com reposição `B = 1000` vezes. O procedimento é pareado: uma comparação entre dois modelos usa os mesmos índices em cada reamostra.

Esses intervalos descrevem a incerteza decorrente da composição deste pequeno teste, **condicionada aos modelos e às predições já obtidas**. Como os modelos não são retreinados, os intervalos não representam variação entre sementes, inicializações ou execuções de GPU.

Assim, “o intervalo da diferença não cruza zero” significa que a diferença é consistente nas reamostragens das notas desta execução; não constitui, isoladamente, uma conclusão geral sobre arquiteturas.


In [ ]:
# ====== 7. Incerteza: bootstrap sobre as 60 notas de teste ======
from numpy.random import default_rng

_cand = [
    ("E1 (regras)", lambda: [r["e1"] for r in test]),
    ("E2 (LLM, teacher)", lambda: [r["e2"] for r in test]),
    ("E2b (LLM local)", lambda: [r["e2b"] if alinhada(r, "e2b") else ["O"] * len(r["tokens"]) for r in test]),
    ("Naive Bayes", lambda: nb_pred),
    ("HMM", lambda: hmm_pred),
    ("Regressão Logística", lambda: base_pred),
    ("CRF", lambda: crf_pred),
    ("BERTimbau", lambda: bert_pred),
    ("BERT + Viterbi", lambda: bert_constr_pred),
    ("BERT + CRF (emissões fixas)", lambda: bert_crf_pred),
    ("BERT + CRF (teacher E2b)", lambda: bert_crf_e2b_pred),
]
preds_por_modelo = {}
for nome, getter in _cand:
    try:
        preds_por_modelo[nome] = getter()
    except NameError:
        pass

def _metricas_idx(idx, y_true, y_pred):
    yt = [y_true[i] for i in idx]
    yp = [y_pred[i] for i in idx]
    return {
        "estrita": strict_f1(yt, yp),
        "relaxada": relaxed_prf(yt, yp)[2],
        "token": token_macro_f1(yt, yp),
    }

def bootstrap_ic(y_true, preds, B=1000, seed=SEED):
    rng = default_rng(seed)
    n = len(y_true)
    linhas = []
    for nome, y_pred in preds.items():
        amostras = {"estrita": [], "relaxada": [], "token": []}
        for _ in range(B):
            idx = rng.integers(0, n, n)
            metricas = _metricas_idx(idx, y_true, y_pred)
            for chave in amostras:
                amostras[chave].append(metricas[chave])
        linha = {"modelo": nome}
        for chave, valores in amostras.items():
            valores = np.asarray(valores)
            linha[chave] = valores.mean()
            linha[chave + "_lo"] = np.percentile(valores, 2.5)
            linha[chave + "_hi"] = np.percentile(valores, 97.5)
        linhas.append(linha)
    return pd.DataFrame(linhas)

ic_df = bootstrap_ic(y_test, preds_por_modelo, B=1000)
print("IC95 por bootstrap das notas, condicionado a esta execucao:\n")
for _, row in ic_df.iterrows():
    print(
        f"{row['modelo']:<31} "
        f"estrita {row['estrita']:.3f} [{row['estrita_lo']:.3f}, {row['estrita_hi']:.3f}] | "
        f"relaxada {row['relaxada']:.3f} [{row['relaxada_lo']:.3f}, {row['relaxada_hi']:.3f}] | "
        f"token {row['token']:.3f} [{row['token_lo']:.3f}, {row['token_hi']:.3f}]"
    )
os.makedirs("resultados", exist_ok=True)
ic_df.round(6).to_csv("resultados/ic_bootstrap.csv", index=False)

def bootstrap_diff(y_true, pred_a, pred_b, metric="estrita", B=1000, seed=SEED):
    metric_fn = {
        "estrita": strict_f1,
        "relaxada": lambda yt, yp: relaxed_prf(yt, yp)[2],
        "token": token_macro_f1,
    }[metric]
    rng = default_rng(seed)
    n = len(y_true)
    diferencas = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        yt = [y_true[i] for i in idx]
        diferencas.append(
            metric_fn(yt, [pred_a[i] for i in idx])
            - metric_fn(yt, [pred_b[i] for i in idx])
        )
    diferencas = np.asarray(diferencas)
    return (
        diferencas.mean(),
        np.percentile(diferencas, 2.5),
        np.percentile(diferencas, 97.5),
    )

pares = [
    ("HMM", "Naive Bayes"),
    ("CRF", "Regressão Logística"),
    ("Regressão Logística", "Naive Bayes"),
    ("CRF", "HMM"),
    ("BERTimbau", "CRF"),
    ("BERT + Viterbi", "BERTimbau"),
    ("BERT + CRF (emissões fixas)", "BERTimbau"),
    ("BERT + CRF (emissões fixas)", "BERT + Viterbi"),
    ("E2 (LLM, teacher)", "BERTimbau"),
    ("E2 (LLM, teacher)", "E2b (LLM local)"),
    ("BERT + CRF (teacher E2b)", "BERT + CRF (emissões fixas)"),
]
print("\nDiferenca pareada de F1 estrita nas reamostragens desta execucao:\n")
for modelo_a, modelo_b in pares:
    if modelo_a in preds_por_modelo and modelo_b in preds_por_modelo:
        media, lo, hi = bootstrap_diff(
            y_test,
            preds_por_modelo[modelo_a],
            preds_por_modelo[modelo_b],
        )
        leitura = "nao cruza 0" if lo > 0 or hi < 0 else "cruza 0"
        print(
            f"  {modelo_a} - {modelo_b:<31}: "
            f"{media:+.3f}  IC95 [{lo:+.3f}, {hi:+.3f}] -> {leitura}"
        )


## 9. Discussão dos resultados

*(Preencher com os números da §8 e da §8.1 desta execução — gold = consenso adjudicado.)*

**Referências e alunos clássicos.** E1 obteve F1 estrita «0,xxx»; os professores E2 e E2b,
«0,xxx» e «0,xxx». Entre os clássicos: NB «0,xxx», HMM «0,xxx», reg. logística «0,xxx»,
CRF «0,xxx». Intervalos pareados relevantes: «HMM−NB», «CRF−logística».

**Ablação neural controlada.** BERT `argmax` «0,xxx» → Viterbi «0,xxx» → CRF «0,xxx»
(mesmas emissões; diferenças refletem o decodificador nesta execução). ICs pareados:
CRF−BERT «[x; y]», CRF−Viterbi «[x; y]».

**Ablação de professor (nova).** BERT+CRF com silver do E2b: «0,xxx» vs «0,xxx» com o E2.
Leitura: «quanto o aluno herda do estilo/qualidade do professor; comparar também com a
referência direta E2b vs gold».

**Classes raras e custo.** «Concentração dos ganhos por tipo; FONTE-URL como
infraestrutura (ver notebook 2); custos por nota desta execução.»

**Incerteza.** O bootstrap reamostra as 60 notas com as predições fixas; não cobre
variação entre sementes/treinos. Gold pequeno; supervisão silver; ver §8.1.

## 10. Conclusão

*(Preencher após a execução.)*

Percurso NB → HMM → logística → CRF → BERTimbau (+ decodificadores estruturados) sobre o
BIO, medido contra o gold **adjudicado**. Conclusões a sustentar com os números: (1) valor
da estrutura de sequência para fronteiras BIO exatas («clássicos»; «ablação de
decodificador»); (2) distância aluno×professor e o custo de inferência local vs API;
(3) efeito do professor (E2 vs E2b) sobre o aluno destilado; (4) limitações (60 notas,
silver, classes raras) e próximos passos (fechamento da adjudicação; múltiplas sementes;
mais exemplos de FONTE textual e QUALIFICADOR).

## Apêndice — Auditoria da seleção de hiperparâmetros (opcional)

As constantes em `BEST` registram a configuração usada no experimento. Com `REPRODUZIR_BUSCA=True`, as células abaixo refazem a seleção no desenvolvimento *silver* sem consultar o conjunto humano.

O objetivo do apêndice é auditar a seleção, não acrescentar novos resultados no teste. Por isso, ele não avalia novamente configurações alternativas no *gold*.

Para o BERT, pequenas diferenças entre execuções de GPU podem alterar o ranking de taxas próximas. A seleção neural deve ser entendida como parte desta execução, não como um ótimo determinístico universal.


In [ ]:
# ====== (Apendice) Auditoria da selecao classica no dev silver ======
if not REPRODUZIR_BUSCA:
    print("Apendice desativado (REPRODUZIR_BUSCA=False).")
else:
    import random as _rnd
    from itertools import product

    _rnd.seed(SEED)
    _silver = train[:]
    _rnd.shuffle(_silver)
    n_dev = max(1, int(0.12 * len(_silver)))
    dev, train_fit = _silver[:n_dev], _silver[n_dev:]
    y_dev = [r["e2"] for r in dev]
    print(f"selecao: train_fit={len(train_fit)} | dev silver={len(dev)}")

    # Naive Bayes e HMM
    hist_nb, hist_hmm = [], []
    for addk, min_count in product([0.01, 0.1, 0.5, 1.0], [1, 2, 3]):
        wid_s, v2i_s, (emis_s, prior_s, init_s, trans_s) = _contagens(
            train_fit, addk, min_count
        )
        nb_dev = [
            [
                LABELS[int((prior_s + emis_s[:, v2i_s[wid_s(w)]]).argmax())]
                for w in r["tokens"]
            ]
            for r in dev
        ]

        def hmm_dev_predict(tokens):
            obs = [v2i_s[wid_s(w)] for w in tokens]
            n_tokens, n_labels = len(obs), len(LABELS)
            dp = np.full((n_tokens, n_labels), -1e18)
            bp = np.zeros((n_tokens, n_labels), dtype=int)
            dp[0] = init_s + emis_s[:, obs[0]]
            for t in range(1, n_tokens):
                scores = dp[t - 1][:, None] + trans_s
                bp[t] = scores.argmax(0)
                dp[t] = scores.max(0) + emis_s[:, obs[t]]
            seq = [int(dp[-1].argmax())]
            for t in range(n_tokens - 1, 0, -1):
                seq.append(int(bp[t, seq[-1]]))
            return [LABELS[i] for i in reversed(seq)]

        hmm_dev = [hmm_dev_predict(r["tokens"]) for r in dev]
        hist_nb.append((strict_f1(y_dev, nb_dev), {"addk": addk, "min_count": min_count}))
        hist_hmm.append((strict_f1(y_dev, hmm_dev), {"addk": addk, "min_count": min_count}))

    hist_nb.sort(key=lambda x: x[0], reverse=True)
    hist_hmm.sort(key=lambda x: x[0], reverse=True)
    print("\nNaive Bayes - 3 melhores:")
    for f1, cfg in hist_nb[:3]:
        print(f"  F1={f1:.3f} {cfg}")
    print("\nHMM - 3 melhores:")
    for f1, cfg in hist_hmm[:3]:
        print(f"  F1={f1:.3f} {cfg}")

    # Regressao logistica
    Xtf = to_feats(train_fit)
    ytf = [r["e2"] for r in train_fit]
    Xdev = to_feats(dev)
    dv_sel = DictVectorizer(sparse=True)
    Xtf_flat = dv_sel.fit_transform([token for seq in Xtf for token in seq])
    ytf_flat = [tag for seq in ytf for tag in seq]

    hist_lr = []
    for C, class_weight in product(
        [0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
        [None, "balanced"],
    ):
        candidate = LogisticRegression(
            max_iter=1000, C=C, class_weight=class_weight
        ).fit(Xtf_flat, ytf_flat)
        pred_dev = [
            list(candidate.predict(dv_sel.transform(seq)))
            for seq in Xdev
        ]
        hist_lr.append((
            strict_f1(y_dev, pred_dev),
            {"C": C, "class_weight": class_weight},
        ))
    hist_lr.sort(key=lambda x: x[0], reverse=True)
    print("\nRegressao logistica - 3 melhores:")
    for f1, cfg in hist_lr[:3]:
        print(f"  F1={f1:.3f} {cfg}")

    # CRF classico
    hist_crf = []
    for c1, c2 in product(
        [0.0, 0.05, 0.1, 0.3, 1.0],
        [0.05, 0.1, 0.3, 1.0],
    ):
        candidate = sklearn_crfsuite.CRF(
            algorithm="lbfgs",
            c1=c1,
            c2=c2,
            max_iterations=120,
            all_possible_transitions=True,
        )
        candidate.fit(Xtf, ytf)
        hist_crf.append((
            strict_f1(y_dev, candidate.predict(Xdev)),
            {"c1": c1, "c2": c2},
        ))
    hist_crf.sort(key=lambda x: x[0], reverse=True)
    print("\nCRF classico - 3 melhores:")
    for f1, cfg in hist_crf[:3]:
        print(f"  F1={f1:.3f} {cfg}")


In [ ]:
# ====== (Apendice) Auditoria da taxa do BERT no dev silver ======
if not REPRODUZIR_BUSCA:
    print("Apendice desativado (REPRODUZIR_BUSCA=False).")
else:
    import numpy as np

    cols = ["tokens", "labels_bio"]
    ds_fit = make_ds(train_fit, "e2").map(
        tok_align, batched=True, remove_columns=cols
    )
    ds_dev = make_ds(dev, "e2").map(
        tok_align, batched=True, remove_columns=cols
    )

    def logits_to_bio(logits_rows, rows):
        word_logits = logits_por_palavra(logits_rows, rows)
        output = []
        for r, (emissions, positions) in zip(rows, word_logits):
            seq = ["O"] * len(r["tokens"])
            for pos, label_id in zip(positions, np.argmax(emissions, axis=1)):
                seq[pos] = id2label[int(label_id)]
            output.append(seq)
        return output

    hist_bert = []
    for lr in [2e-5, 3e-5, 5e-5]:
        candidate_model = AutoModelForTokenClassification.from_pretrained(
            MODEL_NAME,
            num_labels=len(LABELS),
            id2label=id2label,
            label2id=label2id,
        )
        candidate_args = TrainingArguments(
            output_dir=f"bert_sel_{lr:.0e}",
            learning_rate=lr,
            per_device_train_batch_size=16,
            num_train_epochs=BERT_EPOCHS,
            weight_decay=0.01,
            logging_steps=50,
            report_to="none",
            seed=SEED,
        )
        candidate_trainer = Trainer(
            model=candidate_model,
            args=candidate_args,
            train_dataset=ds_fit,
            data_collator=collator,
            processing_class=tok,
        )
        candidate_trainer.train()
        dev_logits = candidate_trainer.predict(ds_dev).predictions
        f1 = strict_f1(y_dev, logits_to_bio(dev_logits, dev))
        hist_bert.append((f1, lr))
        print(f"  lr={lr:.0e} -> F1 estrita dev apos {BERT_EPOCHS} epocas: {f1:.3f}")

    hist_bert.sort(key=lambda x: x[0], reverse=True)
    print(f"\nTaxa vencedora nesta execucao: {hist_bert[0][1]:.0e}")
    print("Compare com BEST['bert_lr']; se divergir, atualize BEST e reexecute o experimento.")
